# 03 · How the ECG digitiser (a CNN) works

Goal: understand, step by step, the neural network that turns a **photo of a paper ECG** into a **signal**.
We use the PhysioNet 2024 challenge winner, *ECG-Digitiser* (BSD-2). Inference only, no training.

**Data used here:** one rendered page from a public PTB-XL record (CC BY 4.0) and one *simulated* phone photo of it.
No patient data. Run the notebook with the digitiser environment (`data/third_party/venv-digitiser`); a GPU helps.

Outputs are stripped before commit; the rendered HTML stays in the git-ignored `data/` folder.

In [ ]:
import os, time
from pathlib import Path
import numpy as np, torch, matplotlib.pyplot as plt, pandas as pd
from PIL import Image, ImageOps
for v in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    os.environ.setdefault(v, "x")
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
%matplotlib inline

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
DATA = Path(os.environ.get("ECG_DATA_DIR", ROOT / "data"))
if not (DATA / "third_party").exists():          # sibling-worktree layout
    DATA = ROOT.parent / "ECG-AI_KG" / "data"
MODEL = DATA / "third_party/ECG-Digitiser/models/M3/nnUNet_results/Dataset500_Signals/nnUNetTrainer__nnUNetPlans__2d"
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, torch.cuda.get_device_name(0) if dev.type == "cuda" else "")
LABELS = ["background", "I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

## 1 · Load the network
nnU-Net wraps a plain **U-Net**: a CNN that first shrinks the image while learning features (*encoder*), then grows it back to full size (*decoder*) to say, **for every pixel, which class it belongs to**.

In [ ]:
pred = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=False, device=dev, verbose=False)
pred.initialize_from_trained_model_folder(str(MODEL), use_folds=("all",), checkpoint_name="checkpoint_final.pth")
net = pred.network.to(dev).eval()
%matplotlib inline
n_params = sum(p.numel() for p in net.parameters())     # parameters() counts shared tensors once
print(f"{type(net).__name__}: {n_params/1e6:.1f} M parameters = {n_params*4/2**20:.0f} MB in fp32")
print("patch size:", pred.configuration_manager.patch_size, "| normalisation:", pred.configuration_manager.normalization_schemes)
print("classes:", LABELS)


def logits_of(x):
    with torch.no_grad():
        o = net(x)
    return (o[0] if isinstance(o, (list, tuple)) else o)[0].float()

## 2 · The architecture, stage by stage
Each *stage* = two blocks of **Conv 3×3 → InstanceNorm → LeakyReLU**. Between stages the encoder halves the resolution (stride-2 convolution) and doubles the channels, up to 512.

In [ ]:
rows = []
H, W = pred.configuration_manager.patch_size
res = [H, W]
for i, st in enumerate(net.encoder.stages):
    convs = [m for m in st.modules() if isinstance(m, torch.nn.Conv2d)]
    stride = convs[0].stride[0]
    if i:
        res = [res[0] // stride, res[1] // stride]
    rows.append(dict(stage=i, resolution=f"{res[0]}x{res[1]}", channels_out=convs[-1].out_channels,
                     stride_in=stride, params_M=round(sum(p.numel() for p in st.parameters()) / 1e6, 3)))
pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ch = [r["channels_out"] for r in rows]
for i, c in enumerate(ch):                       # encoder going down, decoder going up
    ax.bar(i, c, color="#2563eb"); ax.text(i, c + 8, f"{c}", ha="center", fontsize=8)
    ax.text(i, -45, rows[i]["resolution"], ha="center", fontsize=7)
for j, c in enumerate(reversed(ch[:-1])):
    ax.bar(len(ch) + j, c, color="#16a34a"); ax.text(len(ch) + j, c + 8, f"{c}", ha="center", fontsize=8)
ax.set_xticks([]); ax.set_ylabel("channels"); ax.set_ylim(-70, 600)
ax.set_title("U-Net: blue = encoder (image shrinks, more channels), green = decoder (grows back)")
plt.show()

### One block, in detail

In [ ]:
print(net.encoder.stages[0][0].convs[0])
x = np.linspace(-4, 4, 200)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(x, np.where(x > 0, x, 0.01 * x), label="LeakyReLU (used)"); ax[0].plot(x, np.maximum(0, x), "--", label="ReLU")
ax[0].legend(); ax[0].set_title("activation: negative slope 0.01")
k = net.encoder.stages[0][0].convs[0].conv.weight.detach().cpu().numpy()
ax[1].imshow(np.concatenate([k[i, 0] for i in range(8)], axis=1), cmap="RdBu"); ax[1].set_title("first 8 learned 3x3 filters"); ax[1].axis("off")
plt.show()

- **Conv 3×3:** slides a small learned filter over the image; early filters become edge/line detectors.
- **InstanceNorm:** rescales each channel of *each image* to zero mean and unit variance, so the net is less sensitive to overall brightness. It keeps no running statistics, so it behaves the same in training and use.
- **LeakyReLU:** the non-linearity; lets a small signal through for negative inputs.
- **No dropout, no pooling:** downsampling is done by stride-2 convolutions.
- **Last layer:** 1×1 conv to 13 channels, then **softmax**: one probability per class at each pixel.

## 3 · Input: a page and a (simulated) photo
The real pipeline first upscales the image (here to 2800 px wide) so the network sees the grid and the trace at the scale it was trained on, then slides a 1024×1280 window over it.

In [ ]:
page_path = DATA / "photo_pack/page_01_00009_hr.png"
photo_path = next((DATA / "fake_photos/angle").glob("*.jpg"))
TARGET_W = 2800


def load(p):
    return ImageOps.exif_transpose(Image.open(p)).convert("RGB")


def to_gray_array(img):
    g = ImageOps.grayscale(img)                               # the net has ONE input channel
    g = g.resize((TARGET_W, round(g.height * TARGET_W / g.width)), Image.LANCZOS)
    return np.asarray(g, dtype=np.float32)


imgs = {"clean page": load(page_path), "simulated photo (tilted)": load(photo_path)}
gray = {k: to_gray_array(v) for k, v in imgs.items()}
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, (k, v) in zip(ax, imgs.items()):
    a.imshow(v); a.set_title(f"{k}  original {v.size}, network input {gray[k].shape[::-1]}"); a.axis("off")
plt.show()

Colour is thrown away (grayscale). nnU-Net then **z-scores** each window (subtract its mean, divide by its standard deviation). A shadow or glare changes the local brightness; the network was trained on **synthetic** pages and may never have seen that kind of change.

In [ ]:
t0 = time.time()
page = {}
for k, g in gray.items():
    seg, probs = pred.predict_single_npy_array(g[None, None], {"spacing": (999, 1, 1)}, None, None, True)
    probs = np.asarray(probs).reshape(13, *g.shape).astype(np.float16)
    page[k] = dict(img=g, prob=probs, mask=probs.argmax(0))
print(f"whole-page prediction (sliding window) for both images: {time.time() - t0:.0f}s")


def center_crop(k, h=1024, w=1280):
    # a real-scale window around the densest part of the traces
    ys, xs = np.nonzero(page[k]["mask"] > 0)
    cy, cx = int(np.median(ys)), int(np.median(xs))
    y0 = int(np.clip(cy - h // 2, 0, page[k]["img"].shape[0] - h)); x0 = int(np.clip(cx - w // 2, 0, page[k]["img"].shape[1] - w))
    return y0, x0


def crop_tensor(k):
    y0, x0 = center_crop(k)
    c = page[k]["img"][y0:y0 + 1024, x0:x0 + 1280]
    z = (c - c.mean()) / (c.std() + 1e-8)                      # z-score normalisation
    return c, torch.from_numpy(z)[None, None].to(dev), (y0, x0)


c, t, _ = crop_tensor("clean page")
print("tensor shape (batch, channel, H, W):", tuple(t.shape), "| mean/std:", round(t.mean().item(), 3), round(t.std().item(), 3))

## 4 · What the layers see (feature maps)
We record the output of each encoder stage with forward hooks (on one real-scale 1024×1280 window) and show the strongest channels for the same region.

In [ ]:
feats = {}
hooks = [st.register_forward_hook(lambda m, i, o, k=k: feats.__setitem__(k, o.detach())) for k, st in enumerate(net.encoder.stages)]
logits = logits_of(t)
for h in hooks:
    h.remove()
fig, axes = plt.subplots(3, 5, figsize=(18, 9))
axes[0, 0].imshow(c[300:700, 0:500], cmap="gray"); axes[0, 0].set_title("input region", fontsize=8)
axes[1, 0].axis("off"); axes[2, 0].axis("off")
for k in range(4):
    f = feats[k][0].float().cpu(); s = 2 ** k
    top = f.abs().mean((1, 2)).argsort(descending=True)[:3]
    for r, ch_ in enumerate(top):
        a = axes[r, k + 1]
        a.imshow(f[ch_][300 // s:700 // s, 0:500 // s], cmap="magma")
        a.set_title(f"stage {k} ch {int(ch_)} (map {f.shape[1]}x{f.shape[2]})", fontsize=8); a.axis("off")
axes[0, 0].axis("off")
plt.suptitle("Strongest channels per encoder stage: early = edges and ink, later = coarser and more abstract")
plt.show()

## 5 · Output: 13 probabilities per pixel (whole page)

In [ ]:
P = page["clean page"]
mask, prob = P["mask"], P["prob"].astype(np.float32)
conf = prob.max(0)
cmap = plt.get_cmap("tab20", 13)
fig, ax = plt.subplots(1, 3, figsize=(20, 5))
ax[0].imshow(P["img"], cmap="gray"); ax[0].set_title("input (grayscale)")
ax[1].imshow(np.ma.masked_where(mask == 0, mask), cmap=cmap, vmin=0, vmax=12); ax[1].set_title("predicted class per pixel (colour = lead)")
ax[2].imshow(P["img"], cmap="gray")
m = ax[2].imshow(np.ma.masked_where(mask == 0, conf), cmap="viridis", vmin=0.5, vmax=1); ax[2].set_title("confidence on trace pixels")
plt.colorbar(m, ax=ax[2], fraction=0.03)
for a in ax:
    a.axis("off")
plt.show()
print({LABELS[i]: int((mask == i).sum()) for i in range(1, 13)})

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(18, 6))
for a, i in zip(ax.ravel(), [1, 2, 3, 4, 7, 8, 11, 12]):
    a.imshow(prob[i], cmap="inferno", vmin=0, vmax=1); a.set_title(f"P(pixel is lead {LABELS[i]})"); a.axis("off")
plt.show()

## 6 · From mask to signal (simplified)
The real pipeline vectorises with classical code (not a neural net): per column it finds where the trace of each lead is, then converts pixels to mV and seconds with the page scale. Here is the idea for the lead II rhythm strip with a plain column average.

In [ ]:
def trace(mk, lead):
    ys, xs = np.nonzero(mk == LABELS.index(lead))
    cols = np.unique(xs)
    return cols, np.array([ys[xs == c].mean() for c in cols])


fig, ax = plt.subplots(1, 2, figsize=(18, 3.5))
for a, (name, Pk) in zip(ax, page.items()):
    x, y = trace(Pk["mask"], "II")
    a.plot(x, -y, ".", ms=2); a.set_title(f"lead II trace from the mask: {name} ({len(x)} columns found)"); a.set_xlabel("pixel column")
plt.show()

## 7 · Why photos are harder: clean page vs photo

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(20, 9))
for r, (name, Pk) in enumerate(page.items()):
    p = Pk["prob"].astype(np.float32); mk = Pk["mask"]
    ax[r, 0].imshow(Pk["img"], cmap="gray"); ax[r, 0].set_title(name)
    ax[r, 1].imshow(np.ma.masked_where(mk == 0, mk), cmap=cmap, vmin=0, vmax=12); ax[r, 1].set_title(f"mask ({int((mk > 0).sum())} trace pixels)")
    ent = -(p * np.log(p + 1e-9)).sum(0)
    ax[r, 2].imshow(ent, cmap="hot"); ax[r, 2].set_title("uncertainty (entropy): bright = unsure")
for a in ax.ravel():
    a.axis("off")
plt.show()

## 8 · Occlusion test: what does the network depend on?
On the **photo**, hide one square of a real-scale window at a time (paint it with the window's median value) and count how many trace pixels (any lead) the network still finds. A big drop means that region mattered. This is a segmentation-friendly cousin of LIME/SHAP: it needs only forward passes.

In [ ]:
cv, tt, _ = crop_tensor("simulated photo (tilted)")


def trace_pixels(x):
    return int((logits_of(x).argmax(0) > 0).sum())


base = trace_pixels(tt); S = 128; gy, gx = tt.shape[2] // S, tt.shape[3] // S
heat = np.zeros((gy, gx)); fill = tt.median()
t0 = time.time()
for i in range(gy):
    for j in range(gx):
        x = tt.clone(); x[:, :, i * S:(i + 1) * S, j * S:(j + 1) * S] = fill
        heat[i, j] = base - trace_pixels(x)
print(f"{gy * gx} forward passes in {time.time() - t0:.0f}s; trace pixels without occlusion: {base}")
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].imshow(cv, cmap="gray"); ax[0].set_title("photo window")
ax[1].imshow(cv, cmap="gray")
h = ax[1].imshow(heat, cmap="Reds", alpha=0.6, extent=(0, gx * S, gy * S, 0)); plt.colorbar(h, ax=ax[1], fraction=0.03)
ax[1].set_title("trace pixels lost when this square is hidden")
for a in ax:
    a.axis("off")
plt.show()

## What to take away
1. It is a normal CNN (U-Net): 59.2 M parameters, one grayscale channel in, 13 probabilities per pixel out.
2. Everything after the mask (straighten the page, trace to signal, mV scaling) is classic code.
3. Failure on photos means the **masks** go wrong (missing or mixed-up lead pixels); the uncertainty map shows where.
4. When the real photos arrive: re-run sections 5-8 on them and compare with the clean page.